# Trokuti brzina i snaga na Peltonovoj lopatici

**Poglavlje 12: Pokretne lopatice i potisak**

Ovaj interaktivni prikaz nadopunjuje izvod sile i snage na Peltonovoj lopatici. Mijenjanjem apsolutne brzine mlaza, obodne brzine lopatice i izlaznog kuta prati se snaga koju rotor prima.

## Cilj

Na Peltonovoj lopatici relativna brzina mlaza ulazi pod određenim kutem, mijenja smjer i izlazi tako da fluid predaje rotoru tangencijalnu količinu gibanja. Prikaz omogućuje:

1. mijenjanje apsolutne brzine mlaza $c_1$;
2. mijenjanje obodne brzine lopatice $u$;
3. mijenjanje izlaznog kuta $\beta_2$;
4. praćenje snage $P$ i njezine ovisnosti o obodnoj brzini.

## Pretpostavke modela

- jedna reprezentativna lopatica (pojednostavljeni model);
- mlaz dolazi paralelno s osi obodne brzine;
- bez gubitaka u lopatici ($|w_2| = |w_1|$);
- voda gustoće $\rho = 998$ kg/m³, protok mlaza $Q = 0{,}05$ m³/s (pretpostavljen).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

## Računski model

Relativna brzina na ulazu (mlaz minus lopatica):

$$w_1 = c_1 - u.$$

Bez gubitaka u lopatici, iznos relativne brzine zadržava se ($|w_2| = w_1$), ali smjer se mijenja prema izlaznom kutu $\beta_2$ (mjereno od smjera ulaznog $w_1$). Tangencijalna sila na rotor:

$$F_t = \rho Q\,w_1\,(1 - \cos\beta_2).$$

Snaga predana rotoru (sila puta obodna brzina):

$$P = F_t \cdot u = \rho Q\,(c_1 - u)\,u\,(1 - \cos\beta_2).$$

Maksimalna snaga uz idealni $\beta_2 = 180°$ postiže se za $u = c_1/2$.

In [ ]:
RHO = 998.0
Q = 0.05  # m^3/s

def pelton(c1, u, beta2_deg):
    beta2 = np.radians(beta2_deg)
    w1 = c1 - u
    F_t = RHO * Q * w1 * (1 - np.cos(beta2))
    P = F_t * u
    P_max_idealno = RHO * Q * c1**2 / 2  # pri u = c1/2, beta=180
    eta = P / P_max_idealno if P_max_idealno > 0 else 0
    return {'w1': w1, 'F_t': F_t, 'P': P, 'eta': eta}

## Interaktivni prikaz

Klizačima u nastavku biraju se apsolutna brzina mlaza, obodna brzina lopatice i izlazni kut. Lijevi prikaz pokazuje trokute brzina, desni krivulju snage u ovisnosti o obodnoj brzini s istaknutom radnom točkom.

In [ ]:
def pelton_prikaz(c1, u, beta2_deg):
    r = pelton(c1, u, beta2_deg)
    beta2 = np.radians(beta2_deg)

    fig, (ax_tr, ax_P) = plt.subplots(1, 2, figsize=(11, 5))

    # Trokut brzina na ulazu
    ax_tr.annotate('', xy=(c1, 0), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->',
                                      color='#c62828', lw=2.2))
    ax_tr.text(c1/2, -3, f'$c_1$ = {c1:.1f} m/s',
                color='#c62828', ha='center', fontsize=10)
    ax_tr.annotate('', xy=(u, 0), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->',
                                      color='#1565c0', lw=2.2))
    ax_tr.text(u/2, 1.5, f'$u$ = {u:.1f} m/s',
                color='#1565c0', ha='center', fontsize=10)
    ax_tr.annotate('', xy=(c1, 0), xytext=(u, 0),
                    arrowprops=dict(arrowstyle='->',
                                      color='#2e7d32', lw=2.2))
    ax_tr.text((c1 + u)/2, 3, f'$w_1$ = {r["w1"]:.1f} m/s',
                color='#2e7d32', ha='center', fontsize=10)

    ax_tr.set_xlim(-2, max(c1, 1) + 2)
    ax_tr.set_ylim(-8, 8)
    ax_tr.set_xlabel('brzina (m/s)')
    ax_tr.set_title('Trokut brzina na ulazu  '
                     '$\\vec{c}_1 = \\vec{u} + \\vec{w}_1$')
    ax_tr.set_aspect('equal')
    ax_tr.grid(ls=':', alpha=0.5)
    ax_tr.axhline(0, color='gray', lw=0.5)

    # Krivulja P(u) za zadani c1 i beta2
    u_niz = np.linspace(0, c1, 100)
    P_niz = RHO * Q * (c1 - u_niz) * u_niz * (1 - np.cos(beta2))
    ax_P.plot(u_niz, P_niz/1000, color='#1565c0', lw=2.2)
    ax_P.scatter([u], [r['P']/1000], color='#c62828',
                  s=140, zorder=5)
    ax_P.axvline(c1/2, color='gray', ls=':', lw=0.8)
    ax_P.text(c1/2, max(P_niz)*1.05/1000,
               '$u = c_1/2$  (optimum)', color='gray', fontsize=9,
               ha='center')
    ax_P.set_xlabel('obodna brzina  $u$ (m/s)')
    ax_P.set_ylabel('snaga  $P$ (kW)')
    ax_P.set_title(
        f'$\\beta_2$ = {beta2_deg:.0f}°,  '
        f'$P$ = {r["P"]/1000:.2f} kW,  '
        f'$\\eta$ = {r["eta"]*100:.0f}%'
    )
    ax_P.grid(ls=':', alpha=0.5)

    plt.tight_layout()
    plt.show()


interact(
    pelton_prikaz,
    c1=FloatSlider(min=10, max=80, step=2, value=40,
                    description='$c_1$ (m/s)',
                    layout=Layout(width='420px')),
    u=FloatSlider(min=0, max=80, step=1, value=20,
                   description='$u$ (m/s)',
                   layout=Layout(width='420px')),
    beta2_deg=FloatSlider(min=90, max=180, step=2, value=170,
                           description='$\\beta_2$ (°)',
                           layout=Layout(width='420px'))
);

## Pitanja za istraživanje

1. **Optimalna obodna brzina.** Provjeri grafom da maksimum snage pada točno na $u = c_1/2$ neovisno o izlaznom kutu. Zašto upravo polovica brzine mlaza, a ne nula ili $c_1$?

2. **Granica $\beta_2 = 180°$.** Pri potpunom U-okretu (mlaz se vraća unatrag), kakva je teorijska maksimalna snaga? Zašto stvarne lopatice imaju $\beta_2 \approx 165°$, a ne $180°$?

3. **Lopatica u mirovanju.** Pri $u = 0$ (lopatica miruje), kolika je snaga predana rotoru? Zašto, iako sila na lopaticu postoji?

4. **Tehnička procjena.** Za malu hidroelektranu s mlazem $c_1 = 50$ m/s i protokom $Q = 0{,}05$ m³/s, kolika je teorijska maksimalna snaga? Što ograničava stvarni iznos?

## Veza s teorijom poglavlja

Ovaj prikaz materijalizira trokute brzina i radni princip Peltonove turbine iz poglavlja 12. Razdvajanje apsolutne i relativne brzine, te izlazni kut lopatice, izravno određuju snagu predanu rotoru. Ista logika prevodi se na sve akcijske turbine i pumpe — razlika je samo u smjeru prijenosa energije.